# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [66]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [67]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [68]:
# TODO

# Prints the shape
print(df.shape)

# Print the dtype
print(df.dtypes)

# Print the null count per column
print(df.isnull().sum())

# Print the number of exact duplicate rows
print(df.duplicated().sum())

(8, 6)
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
1


**What is wrong with this data?** List at least five specific problems:

1. Item has 1 missing value.
2. Quantity has 1 missing value.
3. Price has 1 missine value.
4. Price is an object, but it should be a float or an int.
5. Ts is an object and is likley meant to be a datetime type.

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [69]:
# Using .duplicated and summing the duplicates
removed = df.duplicated().sum()

# Using the drop_duplicates and copy to copy the "cleaned" df without the duplicates
clean = df.drop_duplicates().copy()

# log('duplicates', 'dropped exact duplicate rows', removed)
log('duplicates', 'dropped exact duplicate rows', removed)

# Assigning clean to the copy without duplicates
clean = clean.copy()

# Printing the clean df shape
print(clean.shape)

[duplicates] dropped exact duplicate rows (1 row(s))
(7, 6)


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [70]:
# Taking the price from the clean df and using str.replace to remove "$". It then uses str.strip() to remove whitespace and casts the values to float with .astype().
clean['price'] = clean['price'].str.replace('$', '', regex=False).str.strip().astype(float)

# Assert the dtype to determine if a "stray" character survived
assert clean['price'].dtype == float

# TODO: log(...) -- note that price arrived as text
log('price', 'coerced to float', len(clean))

[price] coerced to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [71]:
# TODO: clean['qty'] = pd.to_numeric(...)
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()  # Count the NaN values within the 'qty' column
negative = (clean['qty'] < 0).sum()  # Count the values that are less than 0 (negative)

# Drop the rows that have missing data
clean = clean.dropna(subset=['qty']).copy()

# TODO: apply your decision, then log both separately
log('qty_missing', 'dropped rows that had missing information', missing)
log('qty_negative', 'kept negative quantities as refunds', negative)

[qty_missing] dropped rows that had missing information (1 row(s))
[qty_negative] kept negative quantities as refunds (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [72]:
# Print the information from before - unique values from the category variable
print('before:', sorted(clean['category'].unique()))

# Define before count to use for log - how many unique values
before_count = clean['category'].nunique()

# TODO: lowercase, strip, remove punctuation
clean['category'] = (clean['category'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True))

# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {'merch': 'apparel'}

# Replace categories that mean the same thing
clean['category'] = clean['category'].replace(CATEGORY_MAP)

# print('after: ', sorted(clean['category'].unique()))
print('after:', sorted(clean['category'].unique()))
after_count = clean['category'].nunique()

# Include the logs
log('category', 'distinct categories before cleaning', before_count)
log('category', 'distinct categories after cleaning', after_count)

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after: ['apparel', 'food', 'raingear']
[category] distinct categories before cleaning (6 row(s))
[category] distinct categories after cleaning (3 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [73]:
# TODO
# Print the items before cleaning
print('before:', sorted(clean['item'].dropna().unique()))

# Define before count to use for log - how many unique values
before_count = clean['item'].nunique()

# Lowercase, strip, remove punctuation
clean['item'] = (clean['item'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True))

# ITEM_MAP for the judgment calls
ITEM_MAP = {'cheeseburger': 'cheese burger'}

# Replace items that mean the same thing
clean['item'] = clean['item'].replace(ITEM_MAP)

# Print the items after cleaning
print('after:', sorted(clean['item'].dropna().unique()))
after_count = clean['item'].nunique()

# Include the logs
log('item', 'distinct items before cleaning', before_count)
log('item', 'distinct items after cleaning', after_count)

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after: ['cheese burger', 'rain poncho', 'uva tshirt']
[item] distinct items before cleaning (5 row(s))
[item] distinct items after cleaning (3 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [78]:
# TODO

# Parse timestamps into datetimes & coerce failures to NaT
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

# Report how many failed
failed = clean['ts'].isna().sum()

# Add an hour column
clean['hour'] = clean['ts'].dt.hour

# Include log
log('ts', 'parsed timestamps and coerced invalid values to NaT', failed)

[ts] parsed timestamps and coerced invalid values to NaT (3 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [82]:
# TODO: assertions
assert clean['price'].dtype == float # asserts the price is a float
assert clean['qty'].notna().all() # asserts the qty has no na values
assert clean['hour'].dropna().between(0, 23).all() # asserts the non-missing hours are between 0-23
assert clean['category'].isna().sum() == 0 # asserts that the sum of na values is 0
assert 'hour' in clean.columns # asserts hour is within the clean df columns

# TODO: clean['revenue'] = ...
clean['revenue'] = clean['qty'] * clean['price']

# TODO: print rows, units, revenue, distinct categories
print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows: 6
units: 7.0
revenue: 88.5
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [83]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,coerced to float,7
2,qty_missing,dropped rows that had missing information,1
3,qty_negative,kept negative quantities as refunds,1
4,category,distinct categories before cleaning,6
5,category,distinct categories after cleaning,3
6,item,distinct items before cleaning,5
7,item,distinct items after cleaning,3
8,ts,parsed timestamps and coerced invalid values t...,3


In [85]:
# Calculate the current revenue
current_revenue = clean['revenue'].sum()
print('current revenue:', current_revenue)

# Calculate the revenue without including the refund (negative values)
revenue_without_refund = clean.loc[clean['qty'] >= 0, 'revenue'].sum()

# Print metrics to determine the most impactful decision
print('with refund:', current_revenue)
print('without refund:', revenue_without_refund)
print('difference:', current_revenue - revenue_without_refund)

# Calculate the revenue from the duplicate row that was removed
duplicate_revenue = (df[df.duplicated(keep='first')]['qty'] * df[df.duplicated(keep='first')]['price'].str.replace('$', '', regex=False).astype(float)).sum()
print('duplicate revenue:', duplicate_revenue)

current revenue: 88.5
with refund: 88.5
without refund: 106.5
difference: -18.0
duplicate revenue: 15.0


**The decision that mattered most:** was choosing to include the refunds, which are the negative values that we decided to keep in the dataframe.

**Revenue with it:** $88.50

**Revenue without it:** $106.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [90]:
# Checkpoint

rows_after = len(clean)
revenue_after = clean['revenue'].sum()
biggest_decision = 'It was keeping the negative quantities as refunds. This decreased the revenue by $18, compared to if we excluded the refunds from the revenue.' # Which choice moved the number most
revenue_other_way = revenue_without_refund # The total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 6
revenue: 88.5
decision that mattered: It was keeping the negative quantities as refunds. This decreased the revenue by $18, compared to if we excluded the refunds from the revenue.
revenue the other way: 106.5
